# Use case: multi-site variability in ABIDE II with `dashi`

**Scenario.** The Autism Brain Imaging Data Exchange II (ABIDE II) pools subjects recruited and scanned by 19
independent sites. Before analysing such a cohort as a whole, or training a model on it, we want to know *how
comparable the sites are*: do they recruit the same kind of subjects, do they follow the same acquisition
protocol, and does a model learned at one site still work at another?

This notebook answers those questions with the multi-source tools of `dashi`:

| Question | `dashi` tool |
|---|---|
| Does the ASD/control proportion change by site? (**prior shift**) | Data Source Maps (DSMs) + MSV metrics on the label |
| Do the subject characteristics change by site? (**covariate shift**) | Univariate and multivariate DSMs + MSV metrics |
| Do the differences remain within each diagnosis? (**conditional / concept shift**) | Label-conditional DSMs |
| Does a model trained at one site transfer to the others? | Multi-batch models and performance matrices |

Two metrics summarise the DSMs. The **Global Probabilistic Deviation (GPD)** measures how much the site
distributions differ overall (0 = identical, 1 = non-overlapping). The **Source Probabilistic Outlyingness
(SPO)** measures how far each site is from the latent centre of all sites.

**Requirements.** `dashi >= 0.4.3`; the optional cell below installs it. Run the notebook from the `examples` directory.

**Data.** `abide2_phenotypic.csv` holds one row per subject with a subset of the ABIDE II phenotypic file
(Di Martino et al., *Scientific Data* 4, 170010, 2017): acquisition site, diagnosis, age at scan, sex,
handedness category, eye status during the resting-state scan and full-scale IQ (FIQ).

In [ ]:
# Optional: install or update the requirements, then restart the kernel.
# Uncomment the next line to run it.
# %pip install -U "dashi>=0.4.3" nbformat

In [ ]:
from pathlib import Path

import dashi as ds
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42

## 1. Load and format the data

In [ ]:
DATA_PATH = Path("abide2_phenotypic.csv")

raw = pd.read_csv(DATA_PATH)
print(f"{len(raw):,} subjects from {raw['site'].nunique()} sites")
raw.head()

`format_data` casts every column to the type `dashi` expects. The coded variables are declared as categorical
so that no artificial distance is imposed between their codes. In the ABIDE II phenotypic key, sex is coded
1 = male, 2 = female; handedness category 1 = right, 2 = left, 3 = mixed; and eye status 1 = open, 2 = closed.

In [ ]:
SOURCE = "site"
LABEL = "disease_level"
numerical_covariates = ["age_at_scan", "fiq"]
categorical_covariates = ["sex_code", "handedness_category_code", "eye_status_at_scan_code"]
covariates = numerical_covariates + categorical_covariates
analysis_variables = [LABEL] + covariates

data = ds.format_data(
    raw[[SOURCE] + analysis_variables],
    source_column_name=SOURCE,
    numerical_column_names=numerical_covariates,
    categorical_column_names=[SOURCE, LABEL] + categorical_covariates,
)
data.dtypes.rename("dtype").to_frame()

## 2. Cohort composition by site

Site sizes range from about 200 subjects to fewer than 30, and two sites contributed a single diagnosis group.
Keep this in mind throughout: distributions estimated from small sites are less stable.

In [ ]:
site_summary = pd.crosstab(raw[SOURCE], raw[LABEL])
site_summary["n"] = site_summary.sum(axis=1)
site_summary["missing_fiq"] = raw["fiq"].isna().groupby(raw[SOURCE]).sum()
site_summary.sort_values("n", ascending=False)

## 3. Unsupervised characterisation: Data Source Maps

A Data Source Map estimates the probability distribution of a variable in every site over a common support.
Numerical variables are smoothed with a kernel density estimate; categorical variables use their categories.

In [ ]:
source_maps = ds.estimate_univariate_data_source_map(
    data=data,
    source_column=SOURCE,
    numeric_smoothing=True,
)
list(source_maps)

In [ ]:
ds.plot_univariate_data_source_map(
    source_maps, variable_name="age_at_scan", title="Age at scan by site",
).show()

ds.plot_univariate_data_source_map(
    source_maps, variable_name="eye_status_at_scan_code", title="Eye status during the scan by site",
).show()

ds.plot_univariate_data_source_map(
    source_maps, variable_name=LABEL, title="Diagnosis by site",
).show()

## 4. Multi-Source Variability metrics

`estimate_MSV_metrics` embeds the pairwise Jensen-Shannon distances between the site distributions and returns
the GPD of the variable and the SPO of every site.

In [ ]:
def summarize_msv(source_maps, variables):
    """Return the MSV metrics, a GPD ranking and a site-by-variable SPO table."""
    metrics = {}
    gpd_rows = []
    spo_columns = []
    for variable in variables:
        result = ds.estimate_MSV_metrics(source_maps, variable_name=variable)
        metrics[variable] = result
        gpd_rows.append({"variable": variable, "GPD": float(result.GPD)})
        spo_columns.append(pd.Series(result.SPO, index=result.sources, name=variable))
    gpd = pd.DataFrame(gpd_rows).sort_values("GPD", ascending=False, ignore_index=True)
    spo = pd.concat(spo_columns, axis=1).rename_axis("site")
    return metrics, gpd, spo


msv_metrics, gpd, spo = summarize_msv(source_maps, analysis_variables)
gpd.round(3)

**Reading the ranking.**

- **Age at scan** is the most variable covariate (GPD ≈ 0.70): each site recruited a narrow and different age
  range (children only, adolescents, or adults), as the DSM above shows.
- **Eye status** follows (GPD ≈ 0.63). It is not a subject characteristic but an *acquisition protocol*
  decision taken by each site, so the sites split into "eyes open" and "eyes closed" groups.
- **FIQ, handedness and sex** vary moderately (GPD ≈ 0.2-0.3).
- **Diagnosis** is the most stable variable (GPD ≈ 0.18): most sites recruited balanced groups, so prior shift
  is small apart from the two single-diagnosis sites.

The SPO heatmap shows *which* sites drive each result, and the MSV plot places the sites in the latent space of
their distributions: nearby sites have similar distributions and the circle size reflects the sample size.

In [ ]:
fig = px.imshow(
    spo, aspect="auto", color_continuous_scale="Spectral_r",
    labels={"color": "SPO"}, template="plotly_white", title="Source Probabilistic Outlyingness (SPO) by site and variable",
)
fig.update_xaxes(side="top")
fig.show()

In [ ]:
ds.plot_MSV(msv_metrics["age_at_scan"], dimensions=2).show()
ds.plot_MSV(msv_metrics["eye_status_at_scan_code"], dimensions=2).show()

## 5. Joint (multivariate) covariate variability

Univariate maps do not show whether *combinations* of covariates differ. The multivariate DSM first embeds all
covariates with Factor Analysis of Mixed Data (FAMD, suitable for numerical plus categorical variables) and then
estimates each site's density in the reduced space.

Dimensionality reduction needs complete rows, so subjects with a missing covariate are left out and the retained
sample is reported.

In [ ]:
joint_data = data[[SOURCE] + covariates].dropna().copy()
joint_data[SOURCE] = joint_data[SOURCE].cat.remove_unused_categories()
print(f"Complete cases: {len(joint_data):,} of {len(data):,} subjects, "
      f"{joint_data[SOURCE].nunique()} of {data[SOURCE].nunique()} sites")

joint_map = ds.estimate_multivariate_data_source_map(
    data=joint_data,
    source_column_name=SOURCE,
    dim_reduction="FAMD",
    dimensions=2,
    kde_resolution=50,
)

In [ ]:
joint_metrics = ds.estimate_MSV_metrics(joint_map)
print(f"Joint covariate GPD: {joint_metrics.GPD:.3f}")

joint_spo = pd.DataFrame({
    "site": joint_metrics.sources,
    "n": np.asarray(joint_metrics.nBySource).astype(int),
    "SPO": joint_metrics.SPO,
}).sort_values("SPO", ascending=False, ignore_index=True)
joint_spo.round(3)

In [ ]:
ds.plot_multivariate_data_source_map(joint_map).show()
ds.plot_MSV(joint_metrics, dimensions=3).show()

Taken together the covariates separate the sites strongly (joint GPD ≈ 0.82): each site occupies its own region
of the covariate space, which combines its age range with its acquisition protocol.

## 6. Is the variability explained by the diagnosis mix?

Covariate differences between sites could simply reflect a different ASD/control proportion. Label-conditional
DSMs estimate the distributions *within each diagnosis*; `estimate_MSV_metrics` then combines the per-diagnosis
distances. Sites that lack one of the diagnoses cannot be conditioned and are excluded.

In [ ]:
diagnoses_per_site = raw.groupby(SOURCE)[LABEL].nunique()
eligible_sites = diagnoses_per_site[diagnoses_per_site == raw[LABEL].nunique()].index
print("Excluded single-diagnosis sites:", sorted(set(raw[SOURCE]) - set(eligible_sites)))

conditional_data = data[data[SOURCE].isin(eligible_sites)].copy()
conditional_data[SOURCE] = conditional_data[SOURCE].cat.remove_unused_categories()

conditional_maps = ds.estimate_conditional_univariate_data_source_map(
    data=conditional_data,
    source_column_name=SOURCE,
    label_column_name=LABEL,
    numeric_smoothing=True,
)

In [ ]:
_, conditional_gpd, conditional_spo = summarize_msv(conditional_maps, covariates)

comparison = (
    gpd[gpd["variable"].isin(covariates)]
    .rename(columns={"GPD": "GPD_unconditional"})
    .merge(conditional_gpd.rename(columns={"GPD": "GPD_given_diagnosis"}), on="variable")
)
comparison.round(3)

In [ ]:
ds.plot_conditional_univariate_data_source_map(
    conditional_maps, variable_name="age_at_scan",
    title="Age at scan by site, within each diagnosis",
).show()

The GPD of every covariate is practically unchanged after conditioning on the diagnosis. The site differences
are therefore *not* a by-product of the diagnosis mix: ASD and control subjects of the same site resemble each
other more than subjects with the same diagnosis from different sites. The same holds for the joint analysis:

In [ ]:
conditional_joint_data = conditional_data[[SOURCE, LABEL] + covariates].dropna().copy()
conditional_joint_data[SOURCE] = conditional_joint_data[SOURCE].cat.remove_unused_categories()

conditional_joint_maps = ds.estimate_conditional_data_source_map(
    data=conditional_joint_data,
    source_column_name=SOURCE,
    label_column_name=LABEL,
    dim_reduction="FAMD",
    dimensions=2,
    kde_resolution=50,
)

In [ ]:
conditional_joint_metrics = ds.estimate_MSV_metrics(conditional_joint_maps)
print(f"Joint covariate GPD:                 {joint_metrics.GPD:.3f}")
print(f"Joint covariate GPD given diagnosis: {conditional_joint_metrics.GPD:.3f}")

for figure in ds.plot_conditional_data_source_map(conditional_joint_maps):
    figure.show()
ds.plot_MSV(conditional_joint_metrics, dimensions=2).show()

## 7. Supervised characterisation: does a model transfer between sites?

`estimate_multibatch_models` trains one model per site and tests it on a held-out 20% of **every** site. The
diagonal of the resulting matrix is the within-site performance and the off-diagonal cells measure transfer to
other sites.

Here each model is a Random Forest that predicts the diagnosis from the phenotypic covariates. This is a
deliberately modest task (the phenotype carries little diagnostic signal), used to illustrate the workflow.
To keep the test sets meaningful, only sites with both diagnoses and at least `MIN_SITE_SIZE` complete cases
are used.

In [ ]:
def order_for_stratified_split(frame, source_column, label_column):
    """Place a stratified 80% of every site before its remaining 20%.

    dashi tests each model on the last 20% of the rows of every site (no shuffling),
    so this ordering turns that built-in split into a stratified hold-out.
    """
    parts = []
    for _, site_rows in frame.groupby(source_column, sort=True):
        train_index, test_index = train_test_split(
            site_rows.index.to_numpy(), test_size=0.20,
            stratify=site_rows[label_column], random_state=RANDOM_STATE,
        )
        parts.append(site_rows.loc[np.r_[train_index, test_index]])
    return pd.concat(parts, ignore_index=True)

In [ ]:
MIN_SITE_SIZE = 75

supervised_data = raw[[SOURCE, LABEL] + covariates].dropna()
site_sizes = supervised_data.groupby(SOURCE)[LABEL].agg(["size", "nunique"])
selected_sites = site_sizes[(site_sizes["size"] >= MIN_SITE_SIZE) & (site_sizes["nunique"] == 2)].index
supervised_data = supervised_data[supervised_data[SOURCE].isin(selected_sites)]

supervised_data = order_for_stratified_split(supervised_data, SOURCE, LABEL)
supervised_data = ds.format_data(
    supervised_data,
    source_column_name=SOURCE,
    numerical_column_names=numerical_covariates,
    categorical_column_names=[LABEL] + categorical_covariates,
)
pd.crosstab(supervised_data[SOURCE], supervised_data[LABEL], margins=True)

In [ ]:
performance = ds.estimate_multibatch_models(
    data=supervised_data,
    inputs_numerical_column_names=numerical_covariates,
    inputs_categorical_column_names=categorical_covariates,
    output_classification_column_name=LABEL,
    source_column_name=SOURCE,
    learning_strategy="from_scratch",
    model_type="random_forest",
)
print(f"{len(performance)} training-site / test-site combinations evaluated")

In [ ]:
auc = ds.arrange_performance_metrics(metrics=performance, metric_name="AUC_MACRO")
# Rows are labelled 'From <site> to <site>' for single-site training batches.
auc.index = [label.removeprefix("From ").split(" to ")[0] for label in auc.index]
auc = auc.rename_axis(index="training site", columns="test site")
auc.round(2)

In [ ]:
ds.plot_multibatch_performance(
    metrics=performance, metric_name="AUC_MACRO",
    batching_type="source", apply_biclustering_on_source=False,
).show()

ds.plot_multibatch_performance(
    metrics=performance, metric_name="F1-SCORE_MACRO",
    batching_type="source", apply_biclustering_on_source=False,
).show()

In [ ]:
transfer = pd.DataFrame({
    "within_site_AUC": [auc.loc[site, site] for site in auc.index],
    "mean_AUC_on_other_sites": [auc.loc[site].drop(site).mean() for site in auc.index],
}, index=auc.index)
transfer["difference"] = transfer["mean_AUC_on_other_sites"] - transfer["within_site_AUC"]
transfer.round(2)

**Reading the matrix.** Rows are training sites and columns are test sites.

- Performance depends on the *pair* of sites. The GU and OHSU models work on each other's subjects about as well
  as on their own (AUC ≈ 0.8-0.9), but both lose performance on KKI (AUC ≈ 0.55-0.7).
- The KKI model, despite being trained on the largest site, is the weakest one everywhere (AUC ≈ 0.5-0.7).
  More data from a single site does not buy transferability.
- The matrix is not symmetric: transferring from A to B is not the same as from B to A.

The held-out sets contain only about 20 to 40 subjects per site, so individual cells carry wide uncertainty.
The pattern to retain is the spread of values across cells, which a single pooled performance figure would hide.

## 8. Take-home messages

1. **Prior shift is small**: most ABIDE II sites recruited balanced ASD/control groups.
2. **Covariate shift is large** and is led by age at scan (recruitment) and eye status (acquisition protocol).
   Both are site-level design decisions rather than noise.
3. **The shift is not explained by the diagnosis mix**: variability persists within each diagnosis.
4. **Models do not transfer reliably between sites**, so pooled analyses of ABIDE II should treat the site as
   a source of variability (stratification, harmonisation, or site-aware validation such as leave-one-site-out).

GPD and SPO are descriptive distances between estimated distributions, not significance tests. Results for
small sites and rare categories should be read with caution.